In [1]:
import os
print(os.getcwd())

/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/fifth_container/multi_one


In [1]:
import os
import random
from datetime import datetime, timedelta
from moto import mock_aws
import boto3

print("🔧 Step 1: Checking and preparing local workbench folders...")

# A. Create the fleet_data folder if it is missing
if not os.path.exists("fleet_data"):
    os.makedirs("fleet_data")
    print("📁 Created missing 'fleet_data' folder.")

local_raw_path = os.path.join("fleet_data", "raw_fleet_telemetry.csv")

# B. Manufacture a fresh 1,000-row telemetry dataset automatically
print("📝 Manufacturing fresh diagnostic data rows...")
with open(local_raw_path, "w") as f:
    f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n")
    start_time = datetime.now()
    fault_codes = ["P0171", "P0300", "P0420", "NONE", "NONE", "NONE"]
    
    for i in range(1000):
        timestamp = start_time + timedelta(seconds=i)
        v_id = f"VIN-{random.randint(000, 1006)}"
        rpm = random.randint(1500, 3500)
        temp = random.randint(85, 105)
        code = random.choice(fault_codes)
        f.write(f"{timestamp},{v_id},{rpm},{temp},{code}\n")

print(f"✅ Telemetry data file successfully secured at: {local_raw_path}\n")


print("⚡ Step 2: Igniting the Multi-Cloud Local Simulator...")

# Stop any older stuck mock sessions if they are lingering
try:
    aws_mock.stop()
except:
    pass

# Start a fresh AWS Simulation
aws_mock = mock_aws()
aws_mock.start()

# Setup the Simulated AWS S3 Environment
s3_client = boto3.client("s3", region_name="us-east-1")
s3_bucket_name = "fleet-raw-data"
s3_client.create_bucket(Bucket=s3_bucket_name)

# Upload the file we just created into our virtual AWS cloud tank
s3_client.upload_file(local_raw_path, s3_bucket_name, "raw_telemetry.csv")
print("☁️ AWS S3 Simulation: Created bucket 'fleet-raw-data' and uploaded 'raw_telemetry.csv'")


# Setup our Simulated Azure Environment
class MockAzureContainerClient:
    def __init__(self):
        self.stored_blobs = {}
    def upload_blob(self, name, data, overwrite=True):
        self.stored_blobs[name] = data
        print(f"☁️ Microsoft Azure Simulation: Successfully intercepted and saved blob '{name}'!")

class MockBlobServiceClient:
    def get_container_client(self, container_name):
        print(f"☁️ Microsoft Azure Simulation: Connected to container '{container_name}'")
        return MockAzureContainerClient()

# Initialize our simulated Azure connection
azure_client = MockBlobServiceClient()
azure_container = azure_client.get_container_client("fleet-clean-alerts")

print("\n⚙️ Integration Complete: Multi-cloud local simulator is running flawlessly!")

🔧 Step 1: Checking and preparing local workbench folders...
📁 Created missing 'fleet_data' folder.
📝 Manufacturing fresh diagnostic data rows...
✅ Telemetry data file successfully secured at: fleet_data/raw_fleet_telemetry.csv

⚡ Step 2: Igniting the Multi-Cloud Local Simulator...
☁️ AWS S3 Simulation: Created bucket 'fleet-raw-data' and uploaded 'raw_telemetry.csv'
☁️ Microsoft Azure Simulation: Connected to container 'fleet-clean-alerts'

⚙️ Integration Complete: Multi-cloud local simulator is running flawlessly!


In [ ]:
import io
from datetime import datetime

# 1. Upgraded Intake Valve: Streams data out of AWS S3
def s3_telemetry_streamer(bucket, key):
    """Downloads raw file data from AWS S3 and streams it line-by-line."""
    response = s3_client.get_object(Bucket=bucket, Key=key)
    
    # Read the S3 object data as text and turn it into a memory-safe stream
    s3_data = response["Body"].read().decode("utf-8")
    stream_buffer = io.StringIO(s3_data)
    
    # Skip the header row
    header = stream_buffer.readline()
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

# 2. Upgraded Exhaust Valve: Prepares data to be sent to Azure
def azure_bulk_writer(container_client, blob_name, processed_records):
    """Compiles clean data rows and uploads them as a unified blob file to Azure."""
    if not processed_records:
        print("ℹ️ No critical incidents to write to Azure.")
        return
    
    # Create a fresh CSV text block for Azure cloud storage
    csv_buffer = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n"
    
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\n"
        )
    
    # Push the final compiled data directly up to the Azure cloud container
    container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)

# 3. Bringing back our core filtration logic from Phase 2
def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

print("⚙️ Step 3 Complete: Multi-cloud valves and core filter logic defined!")

In [ ]:
import io
from pathlib import Path
from datetime import datetime
import boto3
from azure.storage.blob import BlobServiceClient

# ==========================================
# 1. CONFIGURACIÓN DE RUTAS LOCALES (PATHLIB)
# ==========================================
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOCAL_INPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "input"
LOCAL_OUTPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "output"

def s3_telemetry_streamer(s3_client, bucket, key):
    """Streams records directly out of a live AWS S3 object and mirrors locally."""
    response = s3_client.get_object(Bucket=bucket, Key=key)
    s3_data = response["Body"].read().decode("utf-8")
    
    # --- Modernización Dual: Guardar réplica exacta del archivo crudo de entrada ---
    LOCAL_INPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_raw_file = LOCAL_INPUT_ARCHIVE / f"raw_s3_mirror_{timestamp_str}.csv"
    local_raw_file.write_text(s3_data, encoding="utf-8")
    print(f"💾 [Auditoría Local] Réplica cruda de AWS guardada en: {local_raw_file.relative_to(BASE_DIR)}")

    stream_buffer = io.StringIO(s3_data)
    
    # Skip header
    stream_buffer.readline()
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    """Filters out clean records and tags deviations."""
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def azure_bulk_writer(container_client, blob_name, processed_records):
    """Uploads filtered records directly to Azure and saves an audit copy locally."""
    if not processed_records:
        return
    
    # Encabezado estándar impecable
    headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n"
    csv_buffer = headers
    
    for record in processed_records:
        # Corregido: Removida la coma final para evitar columnas vacías fantasmas
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\n"
        )
    
    # 1. Envío directo a la nube de Azure
    container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)
    
    # 2. --- Modernización Dual: Guardar réplica local del reporte procesado ---
    LOCAL_OUTPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_clean_file = LOCAL_OUTPUT_ARCHIVE / f"clean_alerts_{timestamp_str}.csv"
    local_clean_file.write_text(csv_buffer, encoding="utf-8")
    print(f"💾 [Auditoría Local] Reporte filtrado de Azure guardado en: {local_clean_file.relative_to(BASE_DIR)}")

if __name__ == "__main__":
    import os
    print("🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
    
    # Configuración de variables de entorno de producción
    AWS_BUCKET = os.getenv("AWS_INPUT_BUCKET", "prod-fleet-raw-data")
    AWS_KEY = os.getenv("AWS_INPUT_KEY", "daily_telemetry.csv")
    AZURE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    AZURE_CONTAINER = os.getenv("AZURE_OUTPUT_CONTAINER", "prod-fleet-alerts")
    
    # Modo de simulación local si faltan credenciales en desarrollo
    if not AZURE_CONNECTION_STRING:
        print("⚠️ Advertencia: AZURE_STORAGE_CONNECTION_STRING no detectada.")
        print("❌ El script requiere variables de entorno reales de Cloud para ejecutarse en producción.")
        exit(1)
        
    # Inicialización de clientes Cloud de nivel Enterprise
    s3 = boto3.client("s3")
    azure_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
    azure_container_client = azure_service_client.get_container_client(AZURE_CONTAINER)
    
    print("📥 Connecting to AWS S3 Intake Stream...")
    data_stream = s3_telemetry_streamer(s3, AWS_BUCKET, AWS_KEY)
    
    incidents = []
    for raw_row in data_stream:
        clean_row = telemetry_processor(raw_row)
        if clean_row:
            incidents.append(clean_row)
            
    print(f"📤 Uploading {len(incidents)} processed records to Azure Blob Storage...")
    azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents)
    
    print("🏁 Batch complete! Multi-cloud handoff and local auditing successful.")

In [ ]:
import io
import os
from pathlib import Path
from datetime import datetime
import boto3
from azure.storage.blob import BlobServiceClient

# ==========================================
# 1. CONFIGURACIÓN DE RUTAS LOCALES (PATHLIB)
# ==========================================
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOCAL_INPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "input"
LOCAL_OUTPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "output"

def s3_telemetry_streamer(s3_client, bucket, key, simulation_mode=False):
    """Streams records directly out of AWS S3 or fallback to local simulated data."""
    if simulation_mode:
        print("🛠️  [Simulación] Fabricando flujo de datos local en memoria...")
        # Generamos un set de datos rápido de prueba local
        s3_data = (
            "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
            f"{datetime.now()},VIN-999111,2500,95,P0300\n"
            f"{datetime.now()},VIN-222333,1800,88,NONE\n"
            f"{datetime.now()},VIN-444555,3100,102,P0171\n"
        )
    else:
        response = s3_client.get_object(Bucket=bucket, Key=key)
        s3_data = response["Body"].read().decode("utf-8")
    
    # --- Guardar réplica de auditoría local ---
    LOCAL_INPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_raw_file = LOCAL_INPUT_ARCHIVE / f"raw_s3_mirror_{timestamp_str}.csv"
    local_raw_file.write_text(s3_data, encoding="utf-8")
    print(f"💾 [Auditoría Local] Réplica cruda de entrada guardada en: {local_raw_file.relative_to(BASE_DIR)}")

    stream_buffer = io.StringIO(s3_data)
    stream_buffer.readline()  # Saltar encabezado
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    """Filters out clean records and tags deviations."""
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def azure_bulk_writer(container_client, blob_name, processed_records, simulation_mode=False):
    """Uploads filtered records to Azure or locks them down safely to local disk."""
    if not processed_records:
        return
    
    headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n"
    csv_buffer = headers
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\n"
        )
    
    # 1. Intento de subida a la nube
    if simulation_mode:
        print("☁️  [Simulación] Modo local activo. Se saltó la carga real a Azure.")
    else:
        container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)
        print("☁️  [Cloud] Transmisión exitosa a Azure Blob Storage.")
    
    # 2. Guardar réplica local del reporte procesado con Pathlib
    LOCAL_OUTPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_clean_file = LOCAL_OUTPUT_ARCHIVE / f"clean_alerts_{timestamp_str}.csv"
    local_clean_file.write_text(csv_buffer, encoding="utf-8")
    print(f"💾 [Auditoría Local] Reporte filtrado guardado en: {local_clean_file.relative_to(BASE_DIR)}")

if __name__ == "__main__":
    print("🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
    
    AWS_BUCKET = os.getenv("AWS_INPUT_BUCKET", "prod-fleet-raw-data")
    AWS_KEY = os.getenv("AWS_INPUT_KEY", "daily_telemetry.csv")
    AZURE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    AZURE_CONTAINER = os.getenv("AZURE_OUTPUT_CONTAINER", "prod-fleet-alerts")
    
    # Evaluar si entramos en Modo Simulación debido a falta de llaves de producción
    SIMULACION = False
    if not AZURE_CONNECTION_STRING:
        print("⚠️  Advertencia: Variables de entorno Cloud no detectadas.")
        print("🔄  Activando Modo de Simulación de Infraestructura Local...")
        SIMULACION = True
        
    # Inicialización Condicional de Clientes
    s3, azure_container_client = None, None
    if not SIMULACION:
        s3 = boto3.client("s3")
        azure_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
        azure_container_client = azure_service_client.get_container_client(AZURE_CONTAINER)
    
    print("📥 Connecting to S3 Intake Stream...")
    data_stream = s3_telemetry_streamer(s3, AWS_BUCKET, AWS_KEY, simulation_mode=SIMULACION)
    
    incidents = []
    for raw_row in data_stream:
        clean_row = telemetry_processor(raw_row)
        if clean_row:
            incidents.append(clean_row)
            
    print(f"📤 Preparing handoff for {len(incidents)} processed records...")
    azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents, simulation_mode=SIMULACION)
    
    print("🏁 Batch complete! Multi-cloud architectural loop completed successfully.")

In [ ]:
import io
import os
from pathlib import Path
from datetime import datetime
import boto3
from botocore.config import Config  # 🌟 GOLD NUGGET: Control Core de Infraestructura AWS
from azure.storage.blob import BlobServiceClient

# ==========================================
# 1. CONFIGURACIÓN DE RUTAS LOCALES (PATHLIB)
# ==========================================
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOCAL_INPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "input"
LOCAL_OUTPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "output"

def s3_telemetry_streamer(s3_client, bucket, key, simulation_mode=False):
    """Streams records directly out of AWS S3 or fallback to local simulated data."""
    if simulation_mode:
        print("🛠️  [Simulación] Fabricando flujo de datos local en memoria...")
        # Generamos un set de datos rápido de prueba local
        s3_data = (
            "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
            f"{datetime.now()},VIN-999111,2500,95,P0300\n"
            f"{datetime.now()},VIN-222333,1800,88,NONE\n"
            f"{datetime.now()},VIN-444555,3100,102,P0171\n"
        )
    else:
        response = s3_client.get_object(Bucket=bucket, Key=key)
        s3_data = response["Body"].read().decode("utf-8")
    
    # --- Guardar réplica de auditoría local ---
    LOCAL_INPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_raw_file = LOCAL_INPUT_ARCHIVE / f"raw_s3_mirror_{timestamp_str}.csv"
    local_raw_file.write_text(s3_data, encoding="utf-8")
    print(f"💾 [Auditoría Local] Réplica cruda de entrada guardada en: {local_raw_file.relative_to(BASE_DIR)}")

    stream_buffer = io.StringIO(s3_data)
    stream_buffer.readline()  # Saltar encabezado
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    """Filters out clean records and tags deviations."""
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def azure_bulk_writer(container_client, blob_name, processed_records, simulation_mode=False):
    """Uploads filtered records to Azure or locks them down safely to local disk."""
    if not processed_records:
        return
    
    headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n"
    csv_buffer = headers
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\n"
        )
    
    # 1. Intento de subida a la nube
    if simulation_mode:
        print("☁️  [Simulación] Modo local activo. Se saltó la carga real a Azure.")
    else:
        container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)
        print("☁️  [Cloud] Transmisión exitosa a Azure Blob Storage.")
    
    # 2. Guardar réplica local del reporte procesado con Pathlib
    LOCAL_OUTPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_clean_file = LOCAL_OUTPUT_ARCHIVE / f"clean_alerts_{timestamp_str}.csv"
    local_clean_file.write_text(csv_buffer, encoding="utf-8")
    print(f"💾 [Auditoría Local] Reporte filtrado guardado en: {local_clean_file.relative_to(BASE_DIR)}")

if __name__ == "__main__":
    print("🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
    
    AWS_BUCKET = os.getenv("AWS_INPUT_BUCKET", "prod-fleet-raw-data")
    AWS_KEY = os.getenv("AWS_INPUT_KEY", "daily_telemetry.csv")
    AZURE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    AZURE_CONTAINER = os.getenv("AZURE_OUTPUT_CONTAINER", "prod-fleet-alerts")
    
    # Evaluar si entramos en Modo Simulación debido a falta de llaves de producción
    SIMULACION = False
    if not AZURE_CONNECTION_STRING:
        print("⚠️  Advertencia: Variables de entorno Cloud no detectadas.")
        print("🔄  Activando Modo de Simulación de Infraestructura Local...")
        SIMULACION = True
        
    # Inicialización Condicional de Clientes
    s3, azure_container_client = None, None
    if not SIMULACION:
        # 🌟 INYECCIÓN DE GOLD NUGGET: Boto3 Core Config Timeouts
        aws_enterprise_config = Config(
            connect_timeout=5,      # Máximo 5 segundos para establecer el handshake TCP inicial
            read_timeout=10,        # Máximo 10 segundos de silencio en el socket antes de abortar
            retries={
                'max_attempts': 3,  # Reintentar 3 veces de forma elástica antes de levantar un fallo crítico
                'mode': 'standard'
            }
        )
        
        # Pasamos el blueprint de configuración directamente al inicializar el cliente
        s3 = boto3.client("s3", config=aws_enterprise_config)
        azure_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
        azure_container_client = azure_service_client.get_container_client(AZURE_CONTAINER)
    
    print("📥 Connecting to S3 Intake Stream...")
    data_stream = s3_telemetry_streamer(s3, AWS_BUCKET, AWS_KEY, simulation_mode=SIMULACION)
    
    incidents = []
    for raw_row in data_stream:
        clean_row = telemetry_processor(raw_row)
        if clean_row:
            incidents.append(clean_row)
            
    print(f"📤 Preparing handoff for {len(incidents)} processed records...")
    azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents, simulation_mode=SIMULACION)
    
    print("🏁 Batch complete! Multi-cloud architectural loop completed successfully.") 

In [1]:
# Guardamos el código actualizado con Pathlib y Modo Simulación dentro de una variable de texto
script_actualizado = """import io
import os
from pathlib import Path
from datetime import datetime
import boto3
from botocore.config import Config
from azure.storage.blob import BlobServiceClient

BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOCAL_INPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "input"
LOCAL_OUTPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "output"

def s3_telemetry_streamer(s3_client, bucket, key, simulation_mode=False):
    if simulation_mode:
        print("🛠️  [Simulación] Fabricando flujo de datos local en memoria...")
        s3_data = (
            "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\\n"
            f"{datetime.now()},VIN-999111,2500,95,P0300\\n"
            f"{datetime.now()},VIN-222333,1800,88,NONE\\n"
            f"{datetime.now()},VIN-444555,3100,102,P0171\\n"
        )
    else:
        response = s3_client.get_object(Bucket=bucket, Key=key)
        s3_data = response["Body"].read().decode("utf-8")
    
    LOCAL_INPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_raw_file = LOCAL_INPUT_ARCHIVE / f"raw_s3_mirror_{timestamp_str}.csv"
    local_raw_file.write_text(s3_data, encoding="utf-8")
    print(f"💾 [Auditoría Local] Réplica cruda de entrada guardada en: {local_raw_file.relative_to(BASE_DIR)}")

    stream_buffer = io.StringIO(s3_data)
    stream_buffer.readline()
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def azure_bulk_writer(container_client, blob_name, processed_records, simulation_mode=False):
    if not processed_records:
        return
    
    headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\\n"
    csv_buffer = headers
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\\n"
        )
    
    if simulation_mode:
        print("☁️  [Simulación] Modo local activo. Se saltó la carga real a Azure.")
    else:
        container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)
        print("☁️  [Cloud] Transmisión exitosa a Azure Blob Storage.")
    
    LOCAL_OUTPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_clean_file = LOCAL_OUTPUT_ARCHIVE / f"clean_alerts_{timestamp_str}.csv"
    local_clean_file.write_text(csv_buffer, encoding="utf-8")
    print(f"💾 [Auditoría Local] Reporte filtrado guardado en: {local_clean_file.relative_to(BASE_DIR)}")

if __name__ == '__main__':
    print("🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
    
    AWS_BUCKET = os.getenv("AWS_INPUT_BUCKET", "prod-fleet-raw-data")
    AWS_KEY = os.getenv("AWS_INPUT_KEY", "daily_telemetry.csv")
    AZURE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    AZURE_CONTAINER = os.getenv("AZURE_OUTPUT_CONTAINER", "prod-fleet-alerts")
    
    SIMULACION = False
    if not AZURE_CONNECTION_STRING:
        print("⚠️  Advertencia: Variables de entorno Cloud no detectadas.")
        print("🔄  Activando Modo de Simulación de Infraestructura Local...")
        SIMULACION = True
        
    s3, azure_container_client = None, None
    if not SIMULACION:
        aws_enterprise_config = Config(
            connect_timeout=5,
            read_timeout=10,
            retries={'max_attempts': 3, 'mode': 'standard'}
        )
        s3 = boto3.client("s3", config=aws_enterprise_config)
        azure_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
        azure_container_client = azure_service_client.get_container_client(AZURE_CONTAINER)
    
    print("📥 Connecting to S3 Intake Stream...")
    data_stream = s3_telemetry_streamer(s3, AWS_BUCKET, AWS_KEY, simulation_mode=SIMULACION)
    
    incidents = []
    for raw_row in data_stream:
        clean_row = telemetry_processor(raw_row)
        if clean_row:
            incidents.append(clean_row)
            
    print(f"📤 Preparing handoff for {len(incidents)} processed records...")
    azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents, simulation_mode=SIMULACION)
    
    print("🏁 Batch complete! Multi-cloud architectural loop completed successfully.")
"""

# Escribimos el archivo en el disco local de la Mac
with open("multicloud_batch_worker.py", "w", encoding="utf-8") as f:
    f.write(script_actualizado)

print("💾 ¡EXITO! multicloud_batch_worker.py ha sido estampado en el disco duro.")

💾 ¡EXITO! multicloud_batch_worker.py ha sido estampado en el disco duro.


In [2]:
requirements_bloque = """boto3==1.34.0
azure-storage-blob==12.19.0
botocore==1.34.0
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_bloque)

print("💾 ¡Éxito! requirements.txt estampado en el disco.")

💾 ¡Éxito! requirements.txt estampado en el disco.


In [8]:
dockerfile_bloque = """# Base ligera oficial de Python
FROM python:3.11-slim

# Directorio de trabajo dentro del contenedor
WORKDIR /app

# Copiar requerimientos primero para optimizar la caché de capas de Docker
COPY requirements.txt .

# Instalar las librerías de AWS y Azure
RUN pip install --no-cache-dir -r requirements.txt

# Copiar el script de telemetría que estampamos antes
COPY multicloud_batch_worker.py .

# Crear el punto de montaje para persistir los archivos en tu Mac
VOLUME /app/data

# Comando para ejecutar el pipeline
CMD ["python", "multicloud_batch_worker.py"]
"""

with open("Dockerfile", "w", encoding="utf-8") as f:
    f.write(dockerfile_bloque)

print("💾 ¡Éxito! Dockerfile estampado en el disco.") 

💾 ¡Éxito! Dockerfile estampado en el disco.


In [ ]:
user1s-MacBook-Pro:multi_one admin$ docker build -t fleet-pipeline:v1 .


In [9]:
docker run --rm \
  -e AZURE_STORAGE_CONNECTION_STRING="DefaultEndpointsProtocol=https;AccountName=fake_fleet_storage;AccountKey=1234567890abcdef;EndpointSuffix=core.windows.net" \
  -v "$(pwd)/data:/app/data" \
  fleet-pipeline:v1

SyntaxError: invalid syntax (1902334435.py, line 1)

In [10]:
docker run --rm -e AZURE_STORAGE_CONNECTION_STRING="DefaultEndpointsProtocol=https;AccountName=fake_fleet_storage;AccountKey=1234567890abcdef;EndpointSuffix=core.windows.net" -v "$(pwd)/data:/app/data" fleet-pipeline:v1

SyntaxError: invalid syntax (1229915845.py, line 1)

In [7]:
import os
print(os.getcwd())

/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/fifth_container/multi_one


In [ ]:
🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...
⚠️  Advertencia: Variables de entorno Cloud no detectadas.
🔄  Activando Modo de Simulación de Infraestructura Local...
📥 Connecting to S3 Intake Stream...
🛠️  [Simulación] Fabricando flujo de datos local en memoria...
💾 [Auditoría Local] Réplica cruda de entrada guardada en: data/archive/input/raw_s3_mirror_20260623_174352.csv
📤 Preparing handoff for 2 processed records...
☁️  [Simulación] Modo local activo. Se saltó la carga real a Azure.
💾 [Auditoría Local] Reporte filtrado guardado en: data/archive/output/clean_alerts_20260623_174352.csv
🏁 Batch complete! Multi-cloud architectural loop completed successfully.

In [3]:
# Upgrade mas reciente:
import io
import os
import random
from datetime import datetime, timedelta
from pathlib import Path
import boto3
from moto import mock_aws

# ==========================================
# CONFIGURACIÓN GENERAL Y PATHLIB
# ==========================================
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
FLEET_DATA_DIR = BASE_DIR / "fleet_data"

print("🔧 Step 1: Checking and preparing local workbench folders...")

# Usamos Pathlib para crear la carpeta de forma limpia
FLEET_DATA_DIR.mkdir(parents=True, exist_ok=True)
local_raw_path = FLEET_DATA_DIR / "raw_fleet_telemetry.csv"

# A. Fabricar el dataset de 1,000 filas automáticamente
print("📝 Manufacturing fresh diagnostic data rows...")
with open(local_raw_path, "w", encoding="utf-8") as f:
    f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n")
    start_time = datetime.now()
    fault_codes = ["P0171", "P0300", "P0420", "NONE", "NONE", "NONE"]
    
    for i in range(1000):
        timestamp = start_time + timedelta(seconds=i)
        v_id = f"VIN-{random.randint(100, 1006)}"  # Corregido formato de enteros
        rpm = random.randint(1500, 3500)
        temp = random.randint(85, 105)
        code = random.choice(fault_codes)
        f.write(f"{timestamp},{v_id},{rpm},{temp},{code}\n")

print(f"✅ Telemetry data file successfully secured at: {local_raw_path}\n")

print("⚡ Step 2: Igniting the Multi-Cloud Local Simulator...")

# Iniciar la simulación fresca de AWS Moto
aws_mock = mock_aws()
aws_mock.start()

# Configurar el entorno virtual de AWS S3
s3_client = boto3.client("s3", region_name="us-east-1")
s3_bucket_name = "fleet-raw-data"
s3_client.create_bucket(Bucket=s3_bucket_name)

# Subir nuestro CSV al S3 virtual (Convertimos el Path a string para boto3)
s3_client.upload_file(str(local_raw_path), s3_bucket_name, "raw_telemetry.csv")
print("☁️ AWS S3 Simulation: Created bucket 'fleet-raw-data' and uploaded 'raw_telemetry.csv'")


# ==========================================
# SIMULADORES DE AZURE (MOCKS)
# ==========================================
class MockAzureContainerClient:
    def __init__(self):
        self.stored_blobs = {}
    def upload_blob(self, name, data, overwrite=True):
        self.stored_blobs[name] = data
        print(f"☁️ Microsoft Azure Simulation: Successfully intercepted and saved blob '{name}'!")

class MockBlobServiceClient:
    def __init__(self):
        pass
    def get_container_client(self, container_name):
        print(f"☁️ Microsoft Azure Simulation: Connected to container '{container_name}'")
        return MockAzureContainerClient()
        
    @classmethod
    def from_connection_string(cls, conn_str):
        return cls()

# Inicializar nuestro cliente simulado de Azure
azure_container_client = MockBlobServiceClient().get_container_client("fleet-clean-alerts")

print("⚙️ Integration Complete: Multi-cloud local simulator is running flawlessly!\n")


# ==========================================
# VÁLVULAS DE PROCESAMIENTO (ETL LOGIC)
# ==========================================
# 1. Entrada: Lee desde el cliente S3 virtual
def s3_telemetry_streamer(client, bucket, key):
    """Downloads raw file data from AWS S3 and streams it line-by-line."""
    response = client.get_object(Bucket=bucket, Key=key)
    s3_data = response["Body"].read().decode("utf-8")
    stream_buffer = io.StringIO(s3_data)
    
    # Brincar el encabezado
    stream_buffer.readline()
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

# 2. Transformación: Filtro OBD-II de códigos críticos
def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

# 3. Salida: Escribe hacia el contenedor simulado de Azure
def azure_bulk_writer(container_client, blob_name, processed_records):
    """Compiles clean data rows and uploads them as a unified blob file to Azure."""
    if not processed_records:
        print("ℹ️ No critical incidents to write to Azure.")
        return
    
    csv_buffer = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n"
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\n"
        )
    
    container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)


# ==========================================
# EJECUCIÓN DEL PIPELINE (BATCH PRINCIPAL)
# ==========================================
print("📥 Connecting to AWS S3 Intake Stream...")
# Pasamos las variables correctas que configuramos en el Step 2
data_stream = s3_telemetry_streamer(s3_client, s3_bucket_name, "raw_telemetry.csv")

incidents = []
for raw_row in data_stream:
    clean_row = telemetry_processor(raw_row)
    if clean_row:
        incidents.append(clean_row)
        
print(f"📤 Uploading {len(incidents)} processed records to Azure Blob Storage...")
# Ejecución usando nuestro pipeline e inyectando el mock de Azure
azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents)

print("🏁 Batch complete! Multi-cloud handoff and local auditing successful.")

# Apagar el simulador de AWS al finalizar
aws_mock.stop()

🔧 Step 1: Checking and preparing local workbench folders...
📝 Manufacturing fresh diagnostic data rows...
✅ Telemetry data file successfully secured at: /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/fifth_container/multi_one/fleet_data/raw_fleet_telemetry.csv

⚡ Step 2: Igniting the Multi-Cloud Local Simulator...
☁️ AWS S3 Simulation: Created bucket 'fleet-raw-data' and uploaded 'raw_telemetry.csv'
☁️ Microsoft Azure Simulation: Connected to container 'fleet-clean-alerts'
⚙️ Integration Complete: Multi-cloud local simulator is running flawlessly!

📥 Connecting to AWS S3 Intake Stream...
📤 Uploading 478 processed records to Azure Blob Storage...
☁️ Microsoft Azure Simulation: Successfully intercepted and saved blob 'critical_incidents_report.csv'!
🏁 Batch complete! Multi-cloud handoff and local auditing successful.


In [ ]:
Mas actualizado:


In [4]:
import io
import os
import random
from datetime import datetime, timedelta
from pathlib import Path
import boto3
from moto import mock_aws

# ==========================================
# CONFIGURACIÓN GENERAL Y PATHLIB
# ==========================================
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
FLEET_DATA_DIR = BASE_DIR / "fleet_data"

print("🔧 Step 1: Checking and preparing local workbench folders...")

# Usamos Pathlib para asegurar que la carpeta exista de forma segura
FLEET_DATA_DIR.mkdir(parents=True, exist_ok=True)
local_raw_path = FLEET_DATA_DIR / "raw_fleet_telemetry.csv"

# A. Fabricar el dataset de 1,000 filas automáticamente
print("📝 Manufacturing fresh diagnostic data rows...")
with open(local_raw_path, "w", encoding="utf-8") as f:
    f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n")
    start_time = datetime.now()
    fault_codes = ["P0171", "P0300", "P0420", "NONE", "NONE", "NONE"]
    
    for i in range(1000):
        timestamp = start_time + timedelta(seconds=i)
        v_id = f"VIN-{random.randint(100, 1006)}"
        rpm = random.randint(1500, 3500)
        temp = random.randint(85, 105)
        code = random.choice(fault_codes)
        f.write(f"{timestamp},{v_id},{rpm},{temp},{code}\n")

print(f"✅ Telemetry data file successfully secured at: {local_raw_path}\n")


print("⚡ Step 2: Igniting the Multi-Cloud Local Simulator...")

# Iniciar la simulación fresca de AWS Moto en memoria
aws_mock = mock_aws()
aws_mock.start()

# Configurar el entorno virtual de AWS S3
s3_client = boto3.client("s3", region_name="us-east-1")
s3_bucket_name = "fleet-raw-data"
s3_client.create_bucket(Bucket=s3_bucket_name)

# Subir nuestro CSV al S3 virtual (Convertimos el Path de Pathlib a string)
s3_client.upload_file(str(local_raw_path), s3_bucket_name, "raw_telemetry.csv")
print("☁️ AWS S3 Simulation: Created bucket 'fleet-raw-data' and uploaded 'raw_telemetry.csv'")


# ==========================================
# SIMULADORES DE AZURE (MOCKS)
# ==========================================
class MockAzureContainerClient:
    def __init__(self):
        self.stored_blobs = {}
    def upload_blob(self, name, data, overwrite=True):
        self.stored_blobs[name] = data
        print(f"☁️ Microsoft Azure Simulation: Successfully intercepted and saved blob '{name}'!")

class MockBlobServiceClient:
    def __init__(self):
        pass
    def get_container_client(self, container_name):
        print(f"☁️ Microsoft Azure Simulation: Connected to container '{container_name}'")
        return MockAzureContainerClient()
        
    @classmethod
    def from_connection_string(cls, conn_str):
        return cls()

# Inicializar el cliente simulado de Azure
azure_container_client = MockBlobServiceClient().get_container_client("fleet-clean-alerts")

print("⚙️ Integration Complete: Multi-cloud local simulator is running flawlessly!\n")


# ==========================================
# VÁLVULAS DE PROCESAMIENTO (ETL LOGIC)
# ==========================================

# 1. Entrada: NUEVA versión híbrida con simulación rápida integrada y streaming
def s3_telemetry_streamer(s3_client, bucket, key, simulation_mode=False):
    """Streams records directly out of AWS S3 or fallback to local simulated data."""
    if simulation_mode:
        print("🛠️  [Simulación] Fabricando flujo de datos local en memoria...")
        # Generamos el set corto de prueba en memoria sin requerir AWS S3
        s3_data = (
            "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
            f"{datetime.now()},VIN-999111,2500,95,P0300\n"
            f"{datetime.now()},VIN-222333,1800,88,NONE\n"
            f"{datetime.now()},VIN-444555,3100,102,P0171\n"
        )
    else:
        # Modo estándar: Se conecta y descarga la telemetría del AWS S3 simulado
        response = s3_client.get_object(Bucket=bucket, Key=key)
        s3_data = response["Body"].read().decode("utf-8")
        
    # Procesamiento línea por línea usando un buffer de memoria seguro (io.StringIO)
    stream_buffer = io.StringIO(s3_data)
    
    # Brincar la línea de encabezados (headers)
    stream_buffer.readline()
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

# 2. Transformación: Filtro OBD-II de códigos de falla críticos
def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

# 3. Salida: Compilación final y escritura acumulada hacia Azure
def azure_bulk_writer(container_client, blob_name, processed_records):
    """Compiles clean data rows and uploads them as a unified blob file to Azure."""
    if not processed_records:
        print("ℹ️ No critical incidents to write to Azure.")
        return
    
    # Construcción de encabezado y acumulación por fila usando +=
    csv_buffer = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n"
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\n"
        )
    
    container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)


# ==========================================
# EJECUCIÓN DEL PIPELINE (BATCH PRINCIPAL)
# ==========================================

# --- CONFIGURACIÓN DE PRUEBA DE LA ENTRADA ---
# Puedes cambiar simulation_mode a True si quieres probar solo tus 3 filas locales de memoria.
# Si está en False, leerá de forma automatizada las 1,000 líneas cargadas en el S3 simulado.
USAR_SIMULACION_RAPIDA = False

print("📥 Connecting to AWS S3 Intake Stream...")
# data_stream = s3_telemetry_streamer(
#     s3_client=s3_client, 
#     bucket=s3_bucket_name, 
#     key="raw_telemetry.csv", 
#     simulation_mode=USAR_SIMULACION_RAPIDA
# )
data_stream = s3_telemetry_streamer(None, None, None, simulation_mode=True)
incidents = []
for raw_row in data_stream:
    clean_row = telemetry_processor(raw_row)
    if clean_row:
        incidents.append(clean_row)
        
print(f"📤 Uploading {len(incidents)} processed records to Azure Blob Storage...")
# Ejecución final del reporte compilado hacia nuestro cliente Azure simulado
azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents)

print("🏁 Batch complete! Multi-cloud handoff and local auditing successful.")

# Apagar limpiamente el simulador en memoria al finalizar
aws_mock.stop()

🔧 Step 1: Checking and preparing local workbench folders...
📝 Manufacturing fresh diagnostic data rows...
✅ Telemetry data file successfully secured at: /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/fifth_container/multi_one/fleet_data/raw_fleet_telemetry.csv

⚡ Step 2: Igniting the Multi-Cloud Local Simulator...
☁️ AWS S3 Simulation: Created bucket 'fleet-raw-data' and uploaded 'raw_telemetry.csv'
☁️ Microsoft Azure Simulation: Connected to container 'fleet-clean-alerts'
⚙️ Integration Complete: Multi-cloud local simulator is running flawlessly!

📥 Connecting to AWS S3 Intake Stream...
🛠️  [Simulación] Fabricando flujo de datos local en memoria...
📤 Uploading 2 processed records to Azure Blob Storage...
☁️ Microsoft Azure Simulation: Successfully intercepted and saved blob 'critical_incidents_report.csv'!
🏁 Batch complete! Multi-cloud handoff and local auditing successful.


In [ ]:
# Aquí puedes pasarle None a los primeros parámetros porque no se van a usar
data_stream = s3_telemetry_streamer(None, None, None, simulation_mode=True)